# Mission 3: Retention

**Goal:** teach the model a new task without forgetting the original one task. The original task is plain TinyStories. The new task is TinyStories in an instruction format, where each story starts with header lines such as `Summary:`, `Words:`, and `Features:`. 

Your baseline metric is original-task perplexity after both phases. Your model must also learn the new task: its new-task perplexity (the "new-task ppl" value printed after phase 2) must be 4.2 or lower.

**Constraint:** train for 200 optimizer steps on the original task `common.M3_N_STEPS`, then 500 steps on the new task `common.M3_M_STEPS`.

**What you design:** the training method in both phases.

**What you submit:** your algorithm code, your trained model, and your team benchmark.

Edit only the cells marked **YOUR CODE**. Cells that start with `%%writefile` save your code into the `submission/` folder, which you submit on Gradescope. After editing one of these cells, rerun it.

### 1. Setup

Do the setup steps each time you open the notebook:
1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. **Upload `common.py`.** Open the file browser on the left and upload it. Colab deletes uploaded files when the session ends, so upload it again each session.
3. **Run the cell below.**

`common.py` contains the shared code that staff also use for grading. Do not edit it.

In [ ]:
!pip install -q transformers datasets

import os
assert os.path.exists("common.py"), "Put common.py in the same folder as this notebook."
import common

common.set_seed()
os.makedirs("submission/benchmark_data", exist_ok=True)
print("Device:", common.get_device())

### 2. Load data and the base model

In [ ]:
original, val_original = common.load_tinystories()
new, val_new = common.load_new_task()
print("Example new-task entry:\n" + new[0][:400])

model, tokenizer = common.load_model()


def report(label, m):
    """Print original-task and new-task validation perplexity for model `m`."""
    orig_ppl = common.compute_perplexity(m, tokenizer, val_original)
    new_ppl = common.compute_perplexity(m, tokenizer, val_new)
    print(f"{label:<14} original-task ppl {orig_ppl:7.2f} | new-task ppl {new_ppl:7.2f}")


report("Base model", model)

### 3. Your algorithm

`train_original` runs phase 1 and must take exactly `n_steps` optimizer steps. `train_new` runs phase 2, must take exactly `m_steps` steps, and may also use original-task data. Staff score the model you save in section 5, and may rerun this code to check the step counts.

`common.finetune()` accepts `epochs`, `max_steps`, `time_budget_s`, `lr`, and `batch_size`, or you can write your own training loop. Run `help(common.finetune)` in a cell to see all options.

In [ ]:
%%writefile submission/algorithm.py
import common


def train_original(model, tokenizer, original_texts: list[str], n_steps: int):
    """Phase 1: train on the original task for exactly `n_steps` optimizer steps."""
    # YOUR CODE HERE (currently plain fine-tuning; replace with your method)
    common.finetune(model, tokenizer, original_texts, epochs=100, max_steps=n_steps)
    return model


def train_new(model, tokenizer, new_texts: list[str], original_texts: list[str], m_steps: int):
    """Phase 2: train on the new task for exactly `m_steps` optimizer steps."""
    # YOUR CODE HERE (currently plain fine-tuning on the new task only; replace with your method)
    common.finetune(model, tokenizer, new_texts, epochs=100, max_steps=m_steps)
    return model

In [ ]:
train_original = common.load_function("submission/algorithm.py", "train_original")
train_new = common.load_function("submission/algorithm.py", "train_new")

model, tokenizer = common.load_model()
model = train_original(model, tokenizer, original, common.M3_N_STEPS)
report("After phase 1", model)
model = train_new(model, tokenizer, new, original, common.M3_M_STEPS)
report("After phase 2", model)

### 4. Your team benchmark

Requirements:
- `team_benchmark(model, tokenizer)` returns **one number, where higher is better**
- it returns the same number every time it runs (seed any randomness)
- it finishes within `common.BENCHMARK_TIME_LIMIT_S` seconds and does not change the model's weights
- any data it needs goes in `submission/benchmark_data/`, and none of it may come from the training data loaded above

**Where to put your benchmark data.** Choose whichever is easier:
- **Few examples:** write them as a Python list directly in the cell below.
- **More examples, or data you generate with a script:** save a file (for example, `.json` or `.txt`) in `submission/benchmark_data/` and read it inside `team_benchmark`.

In [ ]:
%%writefile submission/benchmark.py
"""Team benchmark."""
import os
import common

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "benchmark_data")

# Option 1: a short list of examples written here
# EXAMPLES = [
#     'First example...',
#     'Second example...',
# ]

# Option 2: a file in submission/benchmark_data/
# import json
# with open(os.path.join(DATA_DIR, "my_data.json")) as f:
#     EXAMPLES = json.load(f)


def team_benchmark(model, tokenizer) -> float:
    """Evaluate `model` and return one number. Higher must mean better."""
    # YOUR CODE HERE
    raise NotImplementedError("Write your team benchmark.")

In [ ]:
team_benchmark = common.load_function("submission/benchmark.py", "team_benchmark")

base_model, base_tokenizer = common.load_model()
print("Base model:")
common.check_team_benchmark(team_benchmark, base_model, base_tokenizer)

print("Your model:", team_benchmark(model, tokenizer))

### 5. Save your submission

Staff load your model with `common.load_model("submission/model")`, so check that it reloads with the same validation perplexity before submitting (use the cell below to check). If you used LoRA, call `model = model.merge_and_unload()` before saving. If you pruned with `torch.nn.utils.prune`, call `prune.remove(...)` on each pruned layer first.

Run this when you are ready to submit. Then upload the zip file to Gradescope.

Your `submission/` folder should contain:
- `algorithm.py`
- `benchmark.py` and `benchmark_data/`
- `model/`

In [ ]:
common.save_model(model, tokenizer)

# Check that the saved model reloads correctly (should match your validation perplexity above)
reloaded, _ = common.load_model("submission/model")
print(f"Reloaded model validation perplexity: {common.compute_perplexity(reloaded, tokenizer, val):.2f}")

common.package_submission("mission3")